# Vexil private binary QA: redteam

In [ ]:

import os,sys,json,hashlib,tarfile
from pathlib import Path
payload_path=Path('/kaggle/input/datasets/sangtruong/vexil-redteam-train291-binary-20261113/qa_payload.bin')
print('DATASET_PRESENT',payload_path.is_file())
assert payload_path.is_file(), str(payload_path)
assert hashlib.sha256(payload_path.read_bytes()).hexdigest()=='8d9c34c7a4547b6bc29587ffafb4ce34a939b7df90548328b92d5407760c3e8d'
dest=Path('/kaggle/working')
with tarfile.open(payload_path,'r:gz') as tf:
    assert all(m.name.startswith('qa-project/') and '..' not in m.name for m in tf)
    tf.extractall(path=dest,filter='data')
project=dest/'qa-project'
sys.path.insert(0,str(project))
os.chdir(project)
import pandas as pd
manifest=pd.read_csv(project/'cache/vexil_takehome_outputs/match_manifest.csv')
train=manifest.loc[manifest.partition=='train']
assert len(train)==291
for mid in train.match_id:
    key=hashlib.sha256(json.dumps(['/matches/'+str(int(mid)),{}],sort_keys=True).encode()).hexdigest()
    assert (project/'cache/api-cache'/(key+'.json.gz')).is_file(),('missing fixture',int(mid))
import research.robustness_study as rs
rs.CACHE=project/'cache'
rs.OUT=dest/'qa-results'
print('INPUT_TRAIN_ONLY_VERIFIED',len(train))

import research.adversarial_cases as adv
adv.CACHE=rs.CACHE
adv.OUT=rs.OUT
adv.main()
r=json.loads((rs.OUT/'adversarial_qa_receipt.json').read_text())
assert r['total_case_assertions']==12063
assert r['real_http_calls']==0
assert r['validation_103_labels_scored']==0 and r['heldout_106_labels_scored']==0
assert r['diagnostic_notes']['first_negative_gold_slope_minute']==99
print('KAGGLE_12063_REDTEAM_COMPLETE',json.dumps(r,sort_keys=True))
